In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from transformers import (
    AutoModelForCausalLM,
    DynamicCache,
    PretrainedConfig,
    PreTrainedModel,
    GenerationMixin,
)


In [ ]:
import os

In [ ]:
hf_token = "###############"
hf_home = "./cache"
os.environ["HF_HOME"] = hf_home
os.environ["HF_TOKEN"] = hf_token


In [ ]:
OUTPUT_PATH = "./merged"  # folder to store the result in
LORA_MERGE_CACHE = "./cache"  # change if you want to keep these for some reason
CONFIG_YML = "./config.yaml"  # merge configuration file
COPY_TOKENIZER = True  # you want a tokenizer? yeah, that's what i thought
LAZY_UNPICKLE = False  # experimental low-memory model loader
LOW_CPU_MEMORY = False  # enable if you somehow have more VRAM than RAM+swap

In [ ]:
import os
HF_DATASETS_CACHE = os.path.join("hf_cache", "datasets")
TRANSFORMERS_CACHE = os.path.join("hf_cache", "transformers")
os.environ["HF_DATASETS_CACHE"] = HF_DATASETS_CACHE
os.environ["TRANSFORMERS_CACHE"] = TRANSFORMERS_CACHE

In [ ]:
hf_home = os.getenv("HF_HOME", hf_home)
print("Using HF_HOME at:", hf_home)

In [ ]:
# actually do merge
import torch
import yaml

from mergekit.config import MergeConfiguration
from mergekit.merge import MergeOptions, run_merge

with open(CONFIG_YML, "r", encoding="utf-8") as fp:
    merge_config = MergeConfiguration.model_validate(yaml.safe_load(fp))

run_merge(
    merge_config,
    out_path=OUTPUT_PATH,
    options=MergeOptions(
        lora_merge_cache=LORA_MERGE_CACHE,
        cuda=torch.cuda.is_available(),
        copy_tokenizer=COPY_TOKENIZER,
        lazy_unpickle=LAZY_UNPICKLE,
        low_cpu_memory=LOW_CPU_MEMORY,
        allow_crimes=True,
        write_model_card=True
    ),
)
print("Done!")

In [ ]:
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained("merged", trust_remote_code=False)
print("== Config ==")
print("arch:", cfg.model_type)
print("hidden_size:", cfg.hidden_size)
print("num_layers:", cfg.num_hidden_layers)
print("num_heads:", cfg.num_attention_heads)
print("rope_theta:", getattr(cfg, "rope_theta", None))


In [ ]:
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams  

In [ ]:
model = AutoModelForCausalLM.from_pretrained("merged", trust_remote_code=False)



In [ ]:
llm = LLM(model="./merged", dtype="float16",gpu_memory_utilization=0.8,)
prompt = "What is capital of France"
outputs = llm.generate([prompt], SamplingParams(max_tokens=64))
print(outputs[0].outputs[0].text)